# European Migration vs Home Population Map

This notebook visualizes migration statistics in Europe using a **stacked 3D extruded map**.

### Visualization Logic
- **Red Layer (Top)**: Represents the **Immigrant Population**. 
  - *Technically, this is the 'Total Population' layer rendered at the bottom, but since the Blue layer covers the bottom portion, only the top (Immigrant portion) remains visible in Red.*
- **Blue Layer (Bottom)**: Represents the **Native Population**.

Data Source: Aggregated from UN 2024 and Eurostat reports.

In [1]:
import pandas as pd
import geopandas as gpd
import pydeck as pdk
import numpy as np

# 1. Define Data (Source: UN 2024 / Eurostat)
# Counts in Millions
data = {
    "Country": [
        "Germany", "France", "United Kingdom", "Spain", "Italy", 
        "Switzerland", "Austria", "Belgium", "Sweden", "Netherlands",
        "Portugal", "Greece", "Ireland", "Denmark", "Norway", 
        "Finland", "Czechia", "Poland", "Romania", "Hungary"
    ],
    "Immigrants_Count": [
        17750084, 12986757, 9985453, 7477749, 6553671,
        2089797, 2327064, 2349032, 1422583, 1216237,
        992536, 1423964, 1216237, 847475, 903348,
        514432, 1025199, 445726, 370980, 250912
    ],
    "Immigrant_Pct": [
        21.1, 18.6, 14.2, 15.9, 11.0,
        23.1, 25.5, 20.0, 11.8, 6.5,
        9.4, 14.2, 23.1, 14.9, 16.5,
        9.2, 9.5, 1.2, 1.9, 2.6
    ]
}

df = pd.DataFrame(data)

# Calculate Total and Native Populations
# Total = Immigrant_Count / (Pct / 100)
df['Total_Pop'] = df['Immigrants_Count'] / (df['Immigrant_Pct'] / 100)
df['Native_Pop'] = df['Total_Pop'] - df['Immigrants_Count']

# Formatting for display
df['Total_Pop_M'] = (df['Total_Pop'] / 1e6).round(2)
df['Immigrants_M'] = (df['Immigrants_Count'] / 1e6).round(2)
df['Native_Pop_M'] = (df['Native_Pop'] / 1e6).round(2)

print(df.head())

          Country  Immigrants_Count  Immigrant_Pct     Total_Pop  \
0         Germany          17750084           21.1  8.412362e+07   
1          France          12986757           18.6  6.982127e+07   
2  United Kingdom           9985453           14.2  7.032009e+07   
3           Spain           7477749           15.9  4.702987e+07   
4           Italy           6553671           11.0  5.957883e+07   

     Native_Pop  Total_Pop_M  Immigrants_M  Native_Pop_M  
0  6.637354e+07        84.12         17.75         66.37  
1  5.683452e+07        69.82         12.99         56.83  
2  6.033464e+07        70.32          9.99         60.33  
3  3.955212e+07        47.03          7.48         39.55  
4  5.302516e+07        59.58          6.55         53.03  


## 2. Load Geometry

In [2]:
# Load World Map
# Using direct URL to avoid geopandas depreciation warning
url = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
print(f"Loading geometry from {url}...")
world = gpd.read_file(url)

# Merge with Migration Data
# 'ADMIN' usually contains the standard country name in Natural Earth data
merged = world.merge(df, how='inner', left_on='ADMIN', right_on='Country')

# Reproject to EPSG:4326 (Lat/Lon) if not already
merged = merged.to_crs(epsg=4326)

print(f"Merged shapes: {merged.shape}")

Loading geometry from https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip...
Merged shapes: (20, 177)


## 3. Visualization with PyDeck

In [4]:
# Elevation Scale Factor
ELEVATION_SCALE = 0.05

initial_view_state = pdk.ViewState(
    latitude=50.0,
    longitude=10.0,
    zoom=3,
    pitch=45,
    bearing=0
)

# Layer 1: Total Population (Milky Neon Rose Base, but will appear as Top)
total_layer = pdk.Layer(
    "GeoJsonLayer",
    merged,
    opacity=1.0, 
    stroked=True,
    filled=True,
    extruded=True,
    wireframe=True,
    get_elevation="Total_Pop",
    elevation_scale=ELEVATION_SCALE,
    get_fill_color=[255, 126, 190, 220], # Milky neon rose
    get_line_color=[255, 255, 255],
    pickable=True,
    auto_highlight=True,
)

# Layer 2: Native Population (Milky Neon Aqua Front)
native_layer = pdk.Layer(
    "GeoJsonLayer",
    merged,
    opacity=1.0,
    stroked=True,
    filled=True,
    extruded=True,
    wireframe=True,
    get_elevation="Native_Pop",
    elevation_scale=ELEVATION_SCALE,
    get_fill_color=[126, 255, 220, 220], # Milky neon aqua
    get_line_color=[255, 255, 255],
    pickable=True,
    auto_highlight=True,
)

# Plain always-visible country labels
label_data = merged.copy()
label_points = label_data.geometry.representative_point()
label_data['label_lon'] = label_points.x
label_data['label_lat'] = label_points.y
label_data['label_z'] = label_data['Total_Pop'] * ELEVATION_SCALE + 120000
label_data['label_text'] = label_data['Country'] + "\n" + label_data['Immigrant_Pct'].astype(str) + "%"

label_layer = pdk.Layer(
    "TextLayer",
    label_data,
    get_position="[label_lon, label_lat, label_z]",
    get_text="label_text",
    get_color=[255, 244, 180, 245],
    get_size=13,
    size_units=pdk.types.String("pixels"),
    get_text_anchor=pdk.types.String("middle"),
    get_alignment_baseline=pdk.types.String("center"),
    billboard=True,
    pickable=False,
    parameters={"depthTest": False},
)

tooltip = {
    "html": "<b>{Country}</b><br>"
            "Total Pop: {Total_Pop_M} M<br>"
            "<span style='color:#FF7EBE'>Immigrants: {Immigrants_M} M ({Immigrant_Pct}%)</span><br>"
            "<span style='color:#7EFFDC'>Native: {Native_Pop_M} M</span>",
    "style": {
        "backgroundColor": "black",
        "color": "white"
    }
}

r = pdk.Deck(
    layers=[total_layer, native_layer, label_layer], 
    initial_view_state=initial_view_state,
    tooltip=tooltip,
    map_style=pdk.map_styles.CARTO_DARK
)

r.to_html("output/europe_migration_map.html")
print("Map saved to europe_migration_map.html")

Map saved to europe_migration_map.html
